In [7]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# 1. Base embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Training data: query + positive/relevant document chunk
train_examples = [
    InputExample(texts=[
        "What is ARIMA used for?",
        "ARIMA is a time series forecasting model using autoregression, differencing, and moving average terms."
    ]),
    InputExample(texts=[
        "How does SARIMA handle seasonality?",
        "SARIMA extends ARIMA by adding seasonal autoregressive, differencing, and moving average components."
    ]),
    InputExample(texts=[
        "What is cosine similarity in RAG?",
        "Cosine similarity measures how close query and document embeddings are by comparing vector direction."
    ]),
    InputExample(texts=[
        "Why do we chunk documents in RAG?",
        "Documents are split into smaller chunks so retrieval can find the most relevant context for a query."
    ]),
]

# 3. DataLoader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=2
)

# 4. Loss for retrieval-style training
train_loss = losses.MultipleNegativesRankingLoss(model)

# 5. Fine-tune
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=10,
    show_progress_bar=True
)

# 6. Save fine-tuned embedding model
model.save("fine_tuned_rag_embedding_model")

/tmp/ipykernel_7761/3109963446.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("fine_tuned_rag_embedding_model")

documents = [
    "ARIMA is used for time series forecasting.",
    "SARIMA is useful when data has seasonality.",
    "Football is played by two teams.",
    "RAG retrieves relevant chunks before LLM generation."
]

query = "Which model handles seasonal time series?"

doc_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

scores = np.dot(doc_embeddings, query_embedding)

top_k = 2
top_indices = np.argsort(scores)[::-1][:top_k]

for idx in top_indices:
    print(round(scores[idx], 4), "->", documents[idx])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.5017 -> SARIMA is useful when data has seasonality.
0.4911 -> ARIMA is used for time series forecasting.
